# Production Pipeline — KNN Anonymization

Fully self-contained notebook — all pipeline and validation logic lives here (no external `.py` modules).

- **Batch:** `parameter_combinations.csv` → `results/experiment_ranking.csv`
- **Single:** set `RUN_MODE = "single"` → exports to `output/`
- Optional overrides in CONFIG: `DATA_FILE_OVERRIDE`, `TARGET_COL_OVERRIDE`, `ID_COLS_OVERRIDE`
- Feature QI = auto categorical + numerical; target is synthesized separately (Karabo-style)
- **Section 4** reports distribution, rare categories, relationships, IV, AUC-ROC, and model performance

**Run from the project folder** (directory containing the data CSV and parameter grid).


In [1]:
# --- imports & paths ---
import itertools
import json
import time
import tracemalloc
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency, ks_2samp
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, RobustScaler, StandardScaler

GRID_CSV_NAMES = {
    "parameter_combinations.csv",
    "parameter_combinations_filtered.csv",
    "parameter_reference.csv",
}


def resolve_pipeline_root() -> Path:
    candidates = [
        Path.cwd().resolve(),
        Path.cwd().resolve() / "No_target",
    ]
    for p in candidates:
        if p.exists() and (p / "parameter_combinations.csv").exists():
            return p
    for p in candidates:
        if not p.exists():
            continue
        for csv in p.glob("*.csv"):
            if csv.name not in GRID_CSV_NAMES and "ranking" not in csv.name.lower():
                return p
    raise FileNotFoundError("Could not find pipeline root (need a data CSV or parameter_combinations.csv).")


def find_data_csv(root: Path, override: str | None = None) -> Path:
    if override:
        path = root / override
        if not path.exists():
            raise FileNotFoundError(f"DATA_FILE_OVERRIDE not found: {path}")
        return path
    for path in sorted(root.glob("*.csv")):
        if path.name in GRID_CSV_NAMES or "ranking" in path.name.lower():
            continue
        return path
    raise FileNotFoundError(f"No data CSV found in {root}")


PIPELINE_ROOT = resolve_pipeline_root()
PARAMETER_REFERENCE_CSV = PIPELINE_ROOT / "parameter_reference.csv"
PARAMETER_COMBINATIONS_RAW = PIPELINE_ROOT / "parameter_combinations.csv"
PARAMETER_COMBINATIONS_CSV = PARAMETER_COMBINATIONS_RAW
RESULTS_DIR = PIPELINE_ROOT / "results"
OUTPUT_DIR = PIPELINE_ROOT / "output"

ROW_SYNTHESIS_MODE = "donor"
MISSING_LABEL = "Missing"
TVD_THRESHOLD = 0.10
KS_THRESHOLD = 0.10
PASS_RATE_TARGET = 0.85

REQUIRED_FILES = [PARAMETER_COMBINATIONS_RAW]
missing = [str(p) for p in REQUIRED_FILES if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n  " + "\n  ".join(missing))

sns.set_style("whitegrid")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Pipeline root: {PIPELINE_ROOT}")
print(f"Grid: {PARAMETER_COMBINATIONS_RAW.name}")


Pipeline root: C:\Users\admin\OneDrive\Documents\GitHub\KNN_demo\No_target
Grid: parameter_combinations.csv


## CONFIG


In [2]:
# batch | single
RUN_MODE = "batch"

EXPERIMENT_RANKING_CSV = RESULTS_DIR / "experiment_ranking.csv"

RANDOM_STATE = 42
CHECKPOINT_EVERY = 25
BATCH_START = 0
BATCH_LIMIT = None
EXPORT_TOP_TO_OUTPUT = True

# Optional schema overrides (None = auto-detect from CSV)
DATA_FILE_OVERRIDE = None
TARGET_COL_OVERRIDE = None
ID_COLS_OVERRIDE = None

# Single-run manual settings
K_ANONYMITY = 5
K_NEIGHBORS = 15
NUM_WEIGHT = 1.0
CAT_WEIGHT = 1.0
SCALER_METHOD = "robust"
CAT_GEN_METHOD = "weighted_mode"
NUM_GEN_METHOD = "interpolation"
TARGET_GEN_METHOD = "probability"
DISTANCE_MODE = "weighted_sum"
NUM_DISTANCE_METRIC = "euclidean"
CAT_DISTANCE_METRIC = "hamming"
MINKOWSKI_P = 3
RUN_NAME = "Production Run"

pd.DataFrame({
    "setting": [
        "RUN_MODE", "PIPELINE_ROOT", "DATA_FILE_OVERRIDE", "TARGET_COL_OVERRIDE",
        "ID_COLS_OVERRIDE", "ROW_SYNTHESIS_MODE", "PARAMETER_COMBINATIONS_CSV", "BATCH_LIMIT",
    ],
    "value": [
        RUN_MODE, str(PIPELINE_ROOT), DATA_FILE_OVERRIDE, TARGET_COL_OVERRIDE,
        ID_COLS_OVERRIDE, ROW_SYNTHESIS_MODE, str(PARAMETER_COMBINATIONS_CSV), BATCH_LIMIT,
    ],
})


,setting,value
0,RUN_MODE,batch
1,PIPELINE_ROOT,C:\Users\admin\OneDrive\Documents\GitHub\KNN_d...
2,DATA_FILE_OVERRIDE,NaN
3,TARGET_COL_OVERRIDE,NaN
4,ID_COLS_OVERRIDE,NaN
5,ROW_SYNTHESIS_MODE,donor
6,PARAMETER_COMBINATIONS_CSV,C:\Users\admin\OneDrive\Documents\GitHub\KNN_d...
7,BATCH_LIMIT,NaN


# Metrics

In [3]:
PSI_THRESHOLD = 0.25
RARE_CATEGORY_THRESHOLD = 0.05
TARGET_RATE_DRIFT_THRESHOLD = 0.05
AUC_RETENTION_THRESHOLD = 0.80


def calculate_psi(real_values, synth_values, bins: int = 10) -> float:
    real_values = pd.Series(real_values).dropna().astype(float)
    synth_values = pd.Series(synth_values).dropna().astype(float)
    if len(real_values) == 0 or len(synth_values) == 0:
        return float("nan")
    if real_values.nunique() <= 1:
        return 0.0 if synth_values.nunique() <= 1 and synth_values.iloc[0] == real_values.iloc[0] else float("nan")
    quantiles = np.linspace(0, 1, bins + 1)
    edges = np.unique(np.quantile(real_values, quantiles))
    if len(edges) < 3:
        return float("nan")
    real_counts, _ = np.histogram(real_values, bins=edges)
    synth_counts, _ = np.histogram(synth_values, bins=edges)
    real_pct = real_counts / max(1, real_counts.sum())
    synth_pct = synth_counts / max(1, synth_counts.sum())
    epsilon = 1e-6
    return float(np.sum((synth_pct - real_pct) * np.log((synth_pct + epsilon) / (real_pct + epsilon))))


def categorical_distribution_metrics(real_data, synth_data, cat_cols):
    rows = []
    for col in cat_cols:
        real_dist = real_data[col].astype(str).value_counts(normalize=True, dropna=False)
        synth_dist = synth_data[col].astype(str).value_counts(normalize=True, dropna=False)
        all_categories = sorted(set(real_dist.index) | set(synth_dist.index))
        real_aligned = real_dist.reindex(all_categories, fill_value=0)
        synth_aligned = synth_dist.reindex(all_categories, fill_value=0)
        drift = float(0.5 * np.abs(real_aligned - synth_aligned).sum())
        real_categories = set(real_dist.index)
        synth_categories = set(synth_dist.index)
        rows.append({
            "column": col,
            "TVD": round(drift, 4),
            "pass_tvd": drift < TVD_THRESHOLD,
            "real_category_count": len(real_categories),
            "synthetic_category_count": len(synth_categories),
            "category_coverage": round(len(real_categories & synth_categories) / max(1, len(real_categories)), 4),
            "new_synthetic_category_count": len(synth_categories - real_categories),
            "missing_synthetic_category_count": len(real_categories - synth_categories),
            "max_category_proportion_diff": round(float(np.abs(real_aligned - synth_aligned).max()), 4),
        })
    return pd.DataFrame(rows)


def rare_category_metrics(original_df, synthetic_df, cols, threshold=RARE_CATEGORY_THRESHOLD):
    rows = []
    for col in cols:
        orig_freq = original_df[col].value_counts(normalize=True)
        rare_cats = orig_freq[orig_freq < threshold].index
        if len(rare_cats) == 0:
            rows.append({
                "column": col,
                "rare_category_count": 0,
                "original_rare_mass": 0.0,
                "synthetic_rare_mass": 0.0,
                "retention_ratio": float("nan"),
            })
            continue
        synth_freq = synthetic_df[col].value_counts(normalize=True)
        orig_mass = float(orig_freq.loc[rare_cats].sum())
        synth_mass = float(synth_freq.reindex(rare_cats, fill_value=0).sum())
        rows.append({
            "column": col,
            "rare_category_count": int(len(rare_cats)),
            "original_rare_mass": round(orig_mass, 4),
            "synthetic_rare_mass": round(synth_mass, 4),
            "retention_ratio": round(synth_mass / orig_mass, 4) if orig_mass > 0 else float("nan"),
        })
    return pd.DataFrame(rows)


def numerical_distribution_metrics(real_data, synth_data, num_cols):
    rows = []
    for col in num_cols:
        real = pd.to_numeric(real_data[col], errors="coerce").dropna()
        synth = pd.to_numeric(synth_data[col], errors="coerce").dropna()
        if len(real) == 0 or len(synth) == 0:
            continue
        ks_stat = float(ks_2samp(real, synth).statistic) if real.nunique() > 1 and synth.nunique() > 1 else float("nan")
        psi = calculate_psi(real, synth)
        rows.append({
            "column": col,
            "KS_statistic": round(ks_stat, 4) if pd.notna(ks_stat) else np.nan,
            "pass_ks": ks_stat < KS_THRESHOLD if pd.notna(ks_stat) else False,
            "psi": round(psi, 4) if pd.notna(psi) else np.nan,
            "pass_psi": psi < PSI_THRESHOLD if pd.notna(psi) else False,
            "real_mean": round(float(real.mean()), 4),
            "synthetic_mean": round(float(synth.mean()), 4),
            "mean_abs_diff": round(abs(float(real.mean() - synth.mean())), 4),
            "real_median": round(float(real.median()), 4),
            "synthetic_median": round(float(synth.median()), 4),
            "median_abs_diff": round(abs(float(real.median() - synth.median())), 4),
            "real_std": round(float(real.std()), 4),
            "synthetic_std": round(float(synth.std()), 4),
            "std_abs_diff": round(abs(float(real.std() - synth.std())), 4),
        })
    return pd.DataFrame(rows)


def cramers_v(x, y) -> float:
    tbl = pd.crosstab(x.astype(str), y.astype(str))
    if tbl.shape[0] < 2 or tbl.shape[1] < 2:
        return float("nan")
    chi2 = chi2_contingency(tbl)[0]
    n = tbl.sum().sum()
    k = min(tbl.shape) - 1
    return float(np.sqrt(chi2 / (n * k))) if n > 0 and k > 0 else 0.0


def target_relationship_metrics(df_actual, df_out, feature_cols, target_col):
    rows = []
    for col in feature_cols:
        if col == target_col:
            continue
        if col in df_actual.columns and pd.api.types.is_numeric_dtype(df_actual[col]):
            c_a = df_actual[col].astype(float).corr(df_actual[target_col].astype(float))
            c_s = df_out[col].astype(float).corr(df_out[target_col].astype(float))
            if pd.notna(c_a) and pd.notna(c_s):
                rows.append({
                    "label": target_col,
                    "column": col,
                    "metric": "correlation",
                    "actual": round(float(c_a), 4),
                    "synthetic": round(float(c_s), 4),
                    "drift": round(abs(float(c_a) - float(c_s)), 4),
                })
        else:
            v_a = cramers_v(df_actual[col], df_actual[target_col])
            v_s = cramers_v(df_out[col], df_out[target_col])
            rows.append({
                "label": target_col,
                "column": col,
                "metric": "cramers_v",
                "actual": round(v_a, 4),
                "synthetic": round(v_s, 4),
                "drift": round(abs(v_a - v_s), 4),
            })
    return pd.DataFrame(rows)


def numerical_correlation_metrics(real_data, synth_data, num_cols):
    if len(num_cols) < 2:
        return pd.DataFrame()
    real_corr = real_data[num_cols].astype(float).corr()
    synth_corr = synth_data[num_cols].astype(float).corr()
    rows = []
    for c1, c2 in itertools.combinations(num_cols, 2):
        real_val = real_corr.loc[c1, c2]
        synth_val = synth_corr.loc[c1, c2]
        rows.append({
            "feature_1": c1,
            "feature_2": c2,
            "real_corr": round(float(real_val), 4) if pd.notna(real_val) else np.nan,
            "synthetic_corr": round(float(synth_val), 4) if pd.notna(synth_val) else np.nan,
            "abs_corr_diff": round(abs(float(real_val) - float(synth_val)), 4)
            if pd.notna(real_val) and pd.notna(synth_val) else np.nan,
        })
    return pd.DataFrame(rows)


def categorical_pair_relationship_metrics(real_data, synth_data, cat_cols):
    rows = []
    for c1, c2 in itertools.combinations(cat_cols, 2):
        real_v = cramers_v(real_data[c1], real_data[c2])
        synth_v = cramers_v(synth_data[c1], synth_data[c2])
        rows.append({
            "feature_1": c1,
            "feature_2": c2,
            "real_cramers_v": round(real_v, 4) if pd.notna(real_v) else np.nan,
            "synthetic_cramers_v": round(synth_v, 4) if pd.notna(synth_v) else np.nan,
            "abs_cramers_v_diff": round(abs(real_v - synth_v), 4) if pd.notna(real_v) and pd.notna(synth_v) else np.nan,
        })
    return pd.DataFrame(rows)


def categorical_numerical_relationship_metrics(real_data, synth_data, cat_cols, num_cols, max_categories=25):
    rows = []
    for cat in cat_cols:
        if real_data[cat].nunique(dropna=False) > max_categories:
            continue
        for num in num_cols:
            real_group = real_data.groupby(cat)[num].mean()
            synth_group = synth_data.groupby(cat)[num].mean()
            common_groups = sorted(set(real_group.index) & set(synth_group.index))
            if not common_groups:
                continue
            real_values = real_group.reindex(common_groups)
            synth_values = synth_group.reindex(common_groups)
            scale = real_data[num].astype(float).std()
            if pd.isna(scale) or scale == 0:
                scale = 1.0
            mean_group_diff = float(np.mean(np.abs(real_values - synth_values)) / scale)
            rows.append({
                "categorical_column": cat,
                "numerical_column": num,
                "common_category_count": len(common_groups),
                "normalised_group_mean_diff": round(mean_group_diff, 4),
            })
    return pd.DataFrame(rows)


def calc_iv(df, feature, target):
    eps = 1e-6
    target_num = pd.to_numeric(df[target], errors="coerce")
    total_good = int((target_num == 0).sum())
    total_bad = int((target_num == 1).sum())
    iv = 0.0
    for _, sub in df.groupby(feature, dropna=False):
        y = pd.to_numeric(sub[target], errors="coerce")
        good = int((y == 0).sum())
        bad = int((y == 1).sum())
        dist_good = good / max(total_good, 1)
        dist_bad = bad / max(total_bad, 1)
        woe = np.log((dist_good + eps) / (dist_bad + eps))
        iv += (dist_good - dist_bad) * woe
    return float(iv)


def iv_retention_metrics(original_df, synthetic_df, cols, target):
    rows = []
    for col in cols:
        orig_iv = calc_iv(original_df, col, target)
        synth_iv = calc_iv(synthetic_df, col, target)
        rows.append({
            "feature": col,
            "original_iv": round(orig_iv, 4),
            "synthetic_iv": round(synth_iv, 4),
            "retention_ratio": round(synth_iv / orig_iv, 4) if abs(orig_iv) > 1e-9 else np.nan,
            "absolute_delta": round(abs(orig_iv - synth_iv), 4),
        })
    return pd.DataFrame(rows).sort_values("absolute_delta", ascending=False)


def target_rate_metrics(df_actual, df_out, target_col):
    actual_rate = float(pd.to_numeric(df_actual[target_col], errors="coerce").mean())
    synthetic_rate = float(pd.to_numeric(df_out[target_col], errors="coerce").mean())
    drift = abs(actual_rate - synthetic_rate)
    return pd.DataFrame([{
        "target_col": target_col,
        "actual_rate": round(actual_rate, 4),
        "synthetic_rate": round(synthetic_rate, 4),
        "rate_drift": round(drift, 4),
        "pass_rate_drift": drift < TARGET_RATE_DRIFT_THRESHOLD,
    }])


def privacy_metrics(df_actual, df_out, qi_cols, suppressed):
    full_hash_actual = pd.util.hash_pandas_object(df_actual[qi_cols].astype(str), index=False)
    full_hash_out = pd.util.hash_pandas_object(df_out[qi_cols].astype(str), index=False)
    replaced_idx = suppressed[suppressed].index
    replaced_out = df_out.loc[replaced_idx, qi_cols].astype(str)
    replaced_hash = pd.util.hash_pandas_object(replaced_out, index=False)
    synthetic_duplicate_rate = 1.0 - (replaced_hash.nunique() / max(1, len(replaced_hash)))
    exact_match_to_actual_rate = float(replaced_hash.isin(set(full_hash_actual)).mean())
    full_dataset_duplicate_rate = 1.0 - (full_hash_out.nunique() / max(1, len(full_hash_out)))
    replaced_feature_cols = [c for c in df_actual.columns if c not in set(globals().get("ID_COLS", []))]
    exact_match_replaced_rows = len(
        df_out.loc[replaced_idx, replaced_feature_cols].merge(df_actual[replaced_feature_cols], how="inner")
    ) / max(len(replaced_idx), 1)
    return pd.DataFrame([{
        "replaced_rows": int(len(replaced_idx)),
        "replaced_unique_profiles": int(replaced_hash.nunique()),
        "synthetic_duplicate_rate": round(synthetic_duplicate_rate, 6),
        "exact_match_to_actual_rate": round(exact_match_to_actual_rate, 6),
        "full_dataset_duplicate_rate": round(full_dataset_duplicate_rate, 6),
        "replaced_exact_match_rate": round(exact_match_replaced_rows, 6),
    }])


def utility_metrics(df_actual, df_out, feature_cols, target_col, random_state=42):
    y_all = pd.to_numeric(df_actual[target_col], errors="coerce")
    if y_all.nunique(dropna=True) != 2:
        return pd.DataFrame([{"status": "skipped_target_not_binary"}])

    train_idx, test_idx = train_test_split(
        df_actual.index, test_size=0.2, random_state=random_state, stratify=y_all
    )

    def encode_frame(frame):
        out = frame[feature_cols].copy()
        for col in feature_cols:
            if pd.api.types.is_numeric_dtype(out[col]):
                numeric = pd.to_numeric(out[col], errors="coerce")
                out[col] = numeric.fillna(numeric.median())
            else:
                enc = LabelEncoder()
                out[col] = enc.fit_transform(out[col].astype(str).fillna(MISSING_LABEL))
        return out

    X_train = encode_frame(df_actual.loc[train_idx])
    X_test = encode_frame(df_actual.loc[test_idx])
    X_syn_train = encode_frame(df_out.loc[train_idx])
    y_train = y_all.loc[train_idx]
    y_test = y_all.loc[test_idx]

    if y_train.nunique() < 2 or pd.to_numeric(df_out.loc[train_idx, target_col], errors="coerce").nunique() < 2:
        return pd.DataFrame([{"status": "skipped_single_class_in_training"}])

    rows = []
    roc_curves = {}
    for name, X_fit, y_fit in [
        ("real_train", X_train, y_train),
        ("synthetic_train", X_syn_train, pd.to_numeric(df_out.loc[train_idx, target_col], errors="coerce")),
    ]:
        model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=random_state)
        model.fit(X_fit, y_fit)
        proba = model.predict_proba(X_test)[:, 1]
        pred = (proba >= 0.5).astype(int)
        fpr, tpr, _ = roc_curve(y_test, proba)
        roc_curves[name] = {"fpr": fpr.tolist(), "tpr": tpr.tolist()}
        rows.append({
            "training_data": name,
            "auc": round(float(roc_auc_score(y_test, proba)), 4),
            "accuracy": round(float(accuracy_score(y_test, pred)), 4),
            "f1_score": round(float(f1_score(y_test, pred)), 4),
        })
    result = pd.DataFrame(rows)
    real_auc = float(result.loc[result["training_data"] == "real_train", "auc"].iloc[0])
    syn_auc = float(result.loc[result["training_data"] == "synthetic_train", "auc"].iloc[0])
    result["auc_retention_ratio"] = np.nan
    result.loc[result["training_data"] == "synthetic_train", "auc_retention_ratio"] = round(
        syn_auc / real_auc if real_auc > 1e-9 else np.nan, 4
    )
    result.attrs["roc_curves"] = roc_curves
    result.attrs["real_auc"] = real_auc
    result.attrs["synthetic_auc"] = syn_auc
    return result


def _metric_val(summary, key, default):
    val = summary.get(key)
    return default if val is None else float(val)


def build_karabo_scorecard(summary):
    rows = [
        {"area": "Karabo-Categorical", "metric": "mean_category_coverage>=1.0", "value": summary.get("mean_category_coverage"), "pass": _metric_val(summary, "mean_category_coverage", 0.0) >= 1.0},
        {"area": "Karabo-Categorical", "metric": "new_synthetic_categories==0", "value": summary.get("total_new_categories"), "pass": int(summary.get("total_new_categories") or 0) == 0},
        {"area": "Karabo-Numerical", "metric": "mean_PSI<0.25", "value": summary.get("mean_psi"), "pass": _metric_val(summary, "mean_psi", 999.0) < PSI_THRESHOLD},
        {"area": "Karabo-Target", "metric": "target_rate_drift<0.05", "value": summary.get("target_rate_drift"), "pass": _metric_val(summary, "target_rate_drift", 999.0) < TARGET_RATE_DRIFT_THRESHOLD},
        {"area": "Karabo-Utility", "metric": "auc_retention>=0.80", "value": summary.get("auc_retention_ratio"), "pass": _metric_val(summary, "auc_retention_ratio", 0.0) >= AUC_RETENTION_THRESHOLD},
        {"area": "Karabo-Privacy", "metric": "synthetic_duplicate_rate", "value": summary.get("synthetic_duplicate_rate"), "pass": True},
        {"area": "Karabo-Privacy", "metric": "replaced_exact_match_rate<0.001", "value": summary.get("replaced_exact_match_rate"), "pass": _metric_val(summary, "replaced_exact_match_rate", 999.0) < 0.001},
    ]
    return pd.DataFrame(rows)


def compute_karabo_metrics(
    df_actual,
    df_out,
    suppressed,
    *,
    feature_categorical_cols,
    numerical_cols,
    tvd_categorical_cols,
    qi_feature_cols,
    qi_cols,
    target_col,
    random_state=42,
):
    category_metrics = categorical_distribution_metrics(df_actual, df_out, tvd_categorical_cols)
    numeric_metrics = numerical_distribution_metrics(df_actual, df_out, numerical_cols)
    rare_category = rare_category_metrics(df_actual, df_out, tvd_categorical_cols)
    relationship_metrics = (
        target_relationship_metrics(df_actual, df_out, qi_feature_cols, target_col)
        if target_col else pd.DataFrame()
    )
    num_correlation_metrics = numerical_correlation_metrics(df_actual, df_out, numerical_cols)
    cat_pair_metrics = categorical_pair_relationship_metrics(df_actual, df_out, feature_categorical_cols)
    cat_num_metrics = categorical_numerical_relationship_metrics(
        df_actual, df_out, feature_categorical_cols, numerical_cols
    )
    iv_metrics = (
        iv_retention_metrics(df_actual, df_out, qi_feature_cols, target_col)
        if target_col else pd.DataFrame()
    )
    target_metrics = target_rate_metrics(df_actual, df_out, target_col) if target_col else pd.DataFrame()
    privacy = privacy_metrics(df_actual, df_out, qi_cols, suppressed)
    utility = (
        utility_metrics(df_actual, df_out, qi_feature_cols, target_col, random_state=random_state)
        if target_col else pd.DataFrame()
    )

    mean_corr_drift = float(relationship_metrics["drift"].mean()) if len(relationship_metrics) else 0.0
    mean_psi = float(numeric_metrics["psi"].dropna().mean()) if len(numeric_metrics) and "psi" in numeric_metrics else float("nan")
    mean_category_coverage = float(category_metrics["category_coverage"].mean()) if len(category_metrics) else float("nan")
    total_new_categories = int(category_metrics["new_synthetic_category_count"].sum()) if len(category_metrics) else 0
    target_rate_drift = float(target_metrics["rate_drift"].iloc[0]) if len(target_metrics) else float("nan")
    auc_retention_ratio = float("nan")
    real_auc = float("nan")
    synthetic_auc = float("nan")
    roc_curves = {}
    if len(utility) and "auc_retention_ratio" in utility.columns:
        vals = utility["auc_retention_ratio"].dropna()
        if len(vals):
            auc_retention_ratio = float(vals.iloc[0])
    if hasattr(utility, "attrs"):
        real_auc = float(utility.attrs.get("real_auc", float("nan")))
        synthetic_auc = float(utility.attrs.get("synthetic_auc", float("nan")))
        roc_curves = utility.attrs.get("roc_curves", {})

    karabo_summary = {
        "mean_category_coverage": round(mean_category_coverage, 4) if pd.notna(mean_category_coverage) else None,
        "total_new_categories": total_new_categories,
        "mean_psi": round(mean_psi, 4) if pd.notna(mean_psi) else None,
        "mean_num_corr_drift": round(float(num_correlation_metrics["abs_corr_diff"].dropna().mean()), 4)
        if len(num_correlation_metrics) else None,
        "mean_cat_pair_drift": round(float(cat_pair_metrics["abs_cramers_v_diff"].dropna().mean()), 4)
        if len(cat_pair_metrics) else None,
        "mean_cat_num_drift": round(float(cat_num_metrics["normalised_group_mean_diff"].dropna().mean()), 4)
        if len(cat_num_metrics) else None,
        "mean_iv_retention": round(float(iv_metrics["retention_ratio"].dropna().mean()), 4) if len(iv_metrics) else None,
        "target_rate_drift": round(target_rate_drift, 4) if pd.notna(target_rate_drift) else None,
        "real_auc": round(real_auc, 4) if pd.notna(real_auc) else None,
        "synthetic_auc": round(synthetic_auc, 4) if pd.notna(synthetic_auc) else None,
        "auc_retention_ratio": round(auc_retention_ratio, 4) if pd.notna(auc_retention_ratio) else None,
        "synthetic_duplicate_rate": float(privacy["synthetic_duplicate_rate"].iloc[0]) if len(privacy) else None,
        "replaced_exact_match_rate": float(privacy["replaced_exact_match_rate"].iloc[0]) if len(privacy) else None,
        "mean_corr_drift": round(mean_corr_drift, 6),
    }
    karabo_scorecard = build_karabo_scorecard(karabo_summary)

    return {
        "category_metrics": category_metrics,
        "numeric_metrics": numeric_metrics,
        "rare_category_metrics": rare_category,
        "relationship_metrics": relationship_metrics,
        "num_correlation_metrics": num_correlation_metrics,
        "cat_pair_relationship_metrics": cat_pair_metrics,
        "cat_num_relationship_metrics": cat_num_metrics,
        "iv_metrics": iv_metrics,
        "target_metrics": target_metrics,
        "privacy_metrics": privacy,
        "utility_metrics": utility,
        "roc_curves": roc_curves,
        "karabo_scorecard": karabo_scorecard,
        "karabo_summary": karabo_summary,
    }


print("Karabo validation metrics loaded.")

Karabo validation metrics loaded.


## Pipeline functions (all logic lives here)


In [4]:
TARGET_NAME_HINTS = ("churn", "target", "label", "class", "outcome", "flag", "y")
ID_NAME_HINTS = ("id", "uuid", "key")


def _is_binary_target(series: pd.Series) -> bool:
    nums = pd.to_numeric(series, errors="coerce")
    if nums.notna().sum() < len(series) * 0.9:
        return False
    uniq = set(nums.dropna().unique())
    return uniq <= {0, 1} or uniq <= {0.0, 1.0} or (len(uniq) == 2 and max(uniq) <= 1)


def infer_dataset_schema(df: pd.DataFrame, target_col_override=None, id_cols_override=None):
    id_cols = list(id_cols_override or [])
    if not id_cols:
        for col in df.columns:
            lc = col.lower()
            if df[col].nunique(dropna=True) == len(df) and (
                any(h in lc for h in ID_NAME_HINTS) or lc.endswith("_id") or lc == "id"
            ):
                id_cols.append(col)

    pool = [c for c in df.columns if c not in set(id_cols)]
    target_col = target_col_override
    if not target_col:
        binary_cols = [c for c in pool if _is_binary_target(df[c])]
        for col in pool:
            if any(h in col.lower() for h in TARGET_NAME_HINTS) and col in binary_cols:
                target_col = col
                break
        if not target_col and binary_cols:
            target_col = binary_cols[-1]

    feature_pool = [c for c in pool if c != target_col]
    categorical_cols, numerical_cols = [], []
    for col in feature_pool:
        series = df[col]
        if pd.api.types.is_numeric_dtype(series):
            numerical_cols.append(col)
        else:
            categorical_cols.append(col)

    kanon_qi_cols = list(categorical_cols)
    for col in numerical_cols:
        if df[col].nunique(dropna=True) <= 100:
            kanon_qi_cols.append(col)

    generalization = {}
    for col in kanon_qi_cols:
        nums = pd.to_numeric(df[col], errors="coerce")
        if nums.notna().mean() < 0.5:
            continue
        nunique = nums.nunique(dropna=True)
        if nunique <= 1:
            continue
        if nunique > 30:
            generalization[col] = {"type": "quantile_bins", "q": 20}
        elif nunique > 8:
            span = float(nums.max() - nums.min())
            step = max(1, int(round(span / 20)))
            generalization[col] = {"type": "bin_floor", "step": step}

    tvd_categorical_cols = list(categorical_cols)
    if target_col and target_col not in tvd_categorical_cols:
        tvd_categorical_cols.append(target_col)
    qi_feature_cols = categorical_cols + numerical_cols
    qi_cols = qi_feature_cols + ([target_col] if target_col and target_col not in qi_feature_cols else [])

    if target_col and target_col in categorical_cols:
        raise ValueError(f"target column must not be treated as feature categorical: {target_col}")
    missing = [c for c in qi_cols if c not in df.columns]
    if missing:
        raise ValueError(f"inferred QI columns missing from data: {missing}")

    return {
        "id_cols": id_cols,
        "target_col": target_col,
        "relationship_col": target_col,
        "categorical_cols": categorical_cols,
        "numerical_cols": numerical_cols,
        "kanon_qi_cols": kanon_qi_cols,
        "generalization": generalization,
        "tvd_categorical_cols": tvd_categorical_cols,
        "qi_feature_cols": qi_feature_cols,
        "qi_cols": qi_cols,
    }


def apply_schema(schema: dict):
    global ID_COLS, TARGET_COL, RELATIONSHIP_COL, CATEGORICAL_COLS, NUMERICAL_COLS
    global KANON_QI_COLS, GENERALIZATION, TVD_CATEGORICAL_COLS, QI_FEATURE_COLS, QI_COLS
    ID_COLS = schema["id_cols"]
    TARGET_COL = schema["target_col"]
    RELATIONSHIP_COL = schema["relationship_col"]
    CATEGORICAL_COLS = schema["categorical_cols"]
    NUMERICAL_COLS = schema["numerical_cols"]
    KANON_QI_COLS = schema["kanon_qi_cols"]
    GENERALIZATION = schema["generalization"]
    TVD_CATEGORICAL_COLS = schema["tvd_categorical_cols"]
    QI_FEATURE_COLS = schema["qi_feature_cols"]
    QI_COLS = schema["qi_cols"]


def generalize_for_kanonymity(df, qi_cols):
    g = df[qi_cols].copy()
    for col, rule in GENERALIZATION.items():
        if col not in g.columns:
            continue
        if rule["type"] == "bin_floor":
            s = pd.to_numeric(g[col], errors="coerce")
            step = float(rule["step"])
            g[col] = (s // step) * step
        elif rule["type"] == "quantile_bins":
            ranked = pd.to_numeric(g[col], errors="coerce").rank(method="first")
            g[col] = pd.qcut(ranked, q=int(rule.get("q", 20)), duplicates="drop").astype(str)
    str_cols = g.select_dtypes(include=["object", "string"]).columns
    for col in str_cols:
        g[col] = g[col].astype(str).fillna(MISSING_LABEL)
    for col in CATEGORICAL_COLS:
        if col in g.columns:
            g[col] = g[col].astype(str).fillna(MISSING_LABEL)
    return g


def identify_suppressed(df, k_anonymity):
    generalized = generalize_for_kanonymity(df, KANON_QI_COLS)
    class_sizes = generalized.groupby(list(generalized.columns), dropna=False).transform("size")
    suppressed = class_sizes < k_anonymity
    pool_idx = np.where(~suppressed.values)[0]
    synth_idx = np.where(suppressed.values)[0]
    if len(pool_idx) == 0:
        raise ValueError(f"k_anonymity={k_anonymity}: no neighbour pool")
    return suppressed, pool_idx, synth_idx


def fit_preprocessors(df, scaler_method):
    cat_encoders = {}
    for col in CATEGORICAL_COLS:
        enc = LabelEncoder()
        enc.fit(df[col].astype(str).fillna(MISSING_LABEL))
        cat_encoders[col] = enc
    num_medians = df[NUMERICAL_COLS].median()
    num_filled = df[NUMERICAL_COLS].fillna(num_medians).values.astype(float)
    scaler_cls = {"standard": StandardScaler, "minmax": MinMaxScaler, "robust": RobustScaler}[scaler_method]
    num_scaler = scaler_cls().fit(num_filled)
    num_p01 = {c: np.percentile(df[c].dropna(), 1) for c in NUMERICAL_COLS}
    num_p99 = {c: np.percentile(df[c].dropna(), 99) for c in NUMERICAL_COLS}
    X_cat = np.column_stack([
        cat_encoders[c].transform(df[c].astype(str).fillna(MISSING_LABEL)) for c in CATEGORICAL_COLS
    ]) if CATEGORICAL_COLS else np.empty((len(df), 0))
    X_num = num_scaler.transform(num_filled)
    num_ranges = np.ptp(num_filled, axis=0)
    return {
        "cat_encoders": cat_encoders,
        "num_medians": num_medians,
        "num_scaler": num_scaler,
        "num_p01": num_p01,
        "num_p99": num_p99,
        "X_cat": X_cat,
        "X_num": X_num,
        "num_filled": num_filled,
        "num_ranges": num_ranges,
    }


def categorical_pairwise_distance(pool_cat, synth_cat, metric):
    mismatch = pool_cat[np.newaxis, :, :] != synth_cat[:, np.newaxis, :]
    n_cat = pool_cat.shape[1]
    if n_cat == 0:
        return np.zeros((len(synth_cat), len(pool_cat)))
    matches = (~mismatch).sum(axis=2).astype(float)
    if metric == "hamming":
        return np.mean(mismatch, axis=2)
    if metric == "jaccard":
        union = 2.0 * n_cat - matches
        with np.errstate(divide="ignore", invalid="ignore"):
            sim = np.where(union > 0, matches / union, 1.0)
        return 1.0 - sim
    if metric == "overlap":
        return 1.0 - matches / n_cat
    raise ValueError(f"Unknown cat distance: {metric}")


def build_neighbor_cache(cfg, prep, pool_idx, synth_idx):
    pool_cat = prep["X_cat"][pool_idx]
    synth_cat = prep["X_cat"][synth_idx]
    if cfg["distance_mode"] == "gower":
        pool_num = prep["num_filled"][pool_idx]
        synth_num = prep["num_filled"][synth_idx]
        n_synth, n_pool = len(synth_num), len(pool_num)
        total_dist = np.zeros((n_synth, n_pool))
        safe_ranges = np.where(prep["num_ranges"] < 1e-8, 1.0, prep["num_ranges"])
        for j in range(pool_num.shape[1]):
            total_dist += np.abs(pool_num[np.newaxis, :, j] - synth_num[:, np.newaxis, j]) / safe_ranges[j]
        for j in range(pool_cat.shape[1]):
            total_dist += (pool_cat[np.newaxis, :, j] != synth_cat[:, np.newaxis, j]).astype(float)
        total_dist /= max(1, pool_num.shape[1] + pool_cat.shape[1])
    else:
        pool_num = prep["X_num"][pool_idx]
        synth_num = prep["X_num"][synth_idx]
        diff = pool_num[np.newaxis, :, :] - synth_num[:, np.newaxis, :]
        metric = cfg["num_distance_metric"]
        if metric == "euclidean":
            num_dist = np.linalg.norm(diff, axis=2)
        elif metric == "manhattan":
            num_dist = np.sum(np.abs(diff), axis=2)
        elif metric == "minkowski":
            p = int(cfg["minkowski_p"])
            num_dist = np.sum(np.abs(diff) ** p, axis=2) ** (1.0 / p)
        else:
            raise ValueError(metric)
        cat_dist = categorical_pairwise_distance(pool_cat, synth_cat, cfg["cat_distance_metric"])
        total_dist = float(cfg["num_weight"]) * num_dist + float(cfg["cat_weight"]) * cat_dist

    k_cache = min(int(cfg["k_neighbors"]), len(pool_idx))
    cache = {}
    for local_i, global_i in enumerate(synth_idx):
        row_dist = total_dist[local_i]
        idx_local = np.argpartition(row_dist, k_cache - 1)[:k_cache]
        order = idx_local[np.argsort(row_dist[idx_local])]
        cache[int(global_i)] = (row_dist[order], pool_idx[order])
    return cache


def generate_categorical(vals, weights, method, rng):
    if method == "mode":
        return Counter(vals).most_common(1)[0][0]
    if method == "weighted_mode":
        counts = defaultdict(float)
        for v, w in zip(vals, weights):
            counts[str(v)] += w
        return max(counts, key=counts.get)
    if method == "probability":
        return str(rng.choice(vals, p=weights / weights.sum()))
    raise ValueError(method)


def _cast_column_value(col, val, df):
    cat_cols = set(CATEGORICAL_COLS) | ({TARGET_COL} if TARGET_COL else set())
    if col in cat_cols and col in df.columns and pd.api.types.is_integer_dtype(df[col]):
        return int(float(val)) if str(val).replace(".", "", 1).isdigit() else val
    if col in NUMERICAL_COLS and col in df.columns and pd.api.types.is_integer_dtype(df[col]):
        return int(round(val))
    return val


def synthesize_dataset(df, cfg, prep, suppressed, pool_idx, synth_idx, neighbor_cache):
    rng = np.random.default_rng(int(cfg.get("random_state", RANDOM_STATE)))
    df_out = df.copy()
    X_num = prep["X_num"]

    for row_idx in synth_idx:
        row_idx = int(row_idx)
        dists_full, nbrs_full = neighbor_cache[row_idx]
        k = min(int(cfg["k_neighbors"]), len(dists_full))
        dists, nbrs = dists_full[:k], nbrs_full[:k]
        w = 1.0 / (dists + 1e-8)
        w = w / w.sum()
        row_out = {}
        synthesis_mode = str(cfg.get("row_synthesis_mode", ROW_SYNTHESIS_MODE)).lower()

        if synthesis_mode == "donor":
            neighbour_pool = df.loc[nbrs]
            for col in CATEGORICAL_COLS:
                vals = neighbour_pool[col].astype(str).fillna(MISSING_LABEL).values
                row_out[col] = str(rng.choice(vals, p=w))
            synth_scaled = np.zeros(len(NUMERICAL_COLS))
            donor = int(rng.choice(nbrs, p=w))
            for j in range(len(NUMERICAL_COLS)):
                t = float(rng.uniform(0.1, 0.9))
                synth_scaled[j] = X_num[row_idx, j] + t * (X_num[donor, j] - X_num[row_idx, j])
        else:
            for col in CATEGORICAL_COLS:
                vals = df.loc[nbrs, col].astype(str).fillna(MISSING_LABEL).values
                row_out[col] = generate_categorical(vals, w, cfg["cat_gen_method"], rng)
            synth_scaled = np.zeros(len(NUMERICAL_COLS))
            for j in range(len(NUMERICAL_COLS)):
                if cfg["num_gen_method"] == "interpolation":
                    nj = rng.choice(nbrs)
                    t = rng.random()
                    synth_scaled[j] = X_num[row_idx, j] + t * (X_num[nj, j] - X_num[row_idx, j])
                elif cfg["num_gen_method"] == "weighted_mean":
                    synth_scaled[j] = float(np.dot(w, X_num[nbrs, j]))
                else:
                    raise ValueError(cfg["num_gen_method"])

        synth_num = prep["num_scaler"].inverse_transform(synth_scaled.reshape(1, -1)).flatten()
        for j, col in enumerate(NUMERICAL_COLS):
            row_out[col] = float(np.clip(synth_num[j], prep["num_p01"][col], prep["num_p99"][col]))

        if TARGET_COL and TARGET_COL in df.columns:
            target_method = str(cfg.get("target_gen_method", "probability"))
            target_vals = df.loc[nbrs, TARGET_COL].astype(str).fillna(MISSING_LABEL).values
            row_out[TARGET_COL] = generate_categorical(target_vals, w, target_method, rng)

        for col in QI_COLS:
            df_out.loc[row_idx, col] = _cast_column_value(col, row_out[col], df)
    return df_out


def compute_metrics(df_actual, df_out, suppressed, relationship_col=None):
    target_col = relationship_col or TARGET_COL
    karabo = compute_karabo_metrics(
        df_actual, df_out, suppressed,
        feature_categorical_cols=CATEGORICAL_COLS,
        numerical_cols=NUMERICAL_COLS,
        tvd_categorical_cols=TVD_CATEGORICAL_COLS,
        qi_feature_cols=QI_FEATURE_COLS,
        qi_cols=QI_COLS,
        target_col=target_col,
        random_state=RANDOM_STATE,
    )
    category_metrics = karabo["category_metrics"]
    numeric_metrics = karabo["numeric_metrics"]
    tvd_pass_rate = float(category_metrics["pass_tvd"].mean())
    ks_pass_rate = float(numeric_metrics["pass_ks"].mean())
    mean_corr_drift = float(karabo["karabo_summary"]["mean_corr_drift"])
    exact_match_rate = float(karabo["karabo_summary"]["replaced_exact_match_rate"])

    scorecard = pd.DataFrame([
        {"area": "Quality-Categorical", "metric": "tvd_pass_rate>=0.85", "value": round(tvd_pass_rate, 4), "pass": tvd_pass_rate >= PASS_RATE_TARGET},
        {"area": "Quality-Numerical", "metric": "ks_pass_rate>=0.85", "value": round(ks_pass_rate, 4), "pass": ks_pass_rate >= PASS_RATE_TARGET},
        {"area": "Quality-Numerical", "metric": "mean_KS<0.10", "value": round(numeric_metrics["KS_statistic"].mean(), 4), "pass": numeric_metrics["KS_statistic"].mean() < KS_THRESHOLD},
        {"area": "Relationships", "metric": "mean_corr_drift<0.05", "value": round(mean_corr_drift, 4), "pass": mean_corr_drift < 0.05},
        {"area": "Privacy", "metric": "replaced_exact_match_rate<0.001", "value": round(exact_match_rate, 6), "pass": exact_match_rate < 0.001},
        {"area": "Suppression", "metric": "recovery_rate", "value": 1.0, "pass": suppressed.sum() > 0},
    ])
    overall_pass = bool(scorecard["pass"].all())
    distribution_summary = build_distribution_summary(df_actual, df_out)

    return {
        **karabo,
        "scorecard": scorecard,
        "distribution_summary": distribution_summary,
        "overall_pass": overall_pass,
        "tvd_pass_rate": round(tvd_pass_rate, 4),
        "ks_pass_rate": round(ks_pass_rate, 4),
        "mean_tvd": round(category_metrics["TVD"].mean(), 4),
        "mean_ks": round(numeric_metrics["KS_statistic"].mean(), 4),
        "mean_corr_drift": round(mean_corr_drift, 6),
        "exact_match_rate": round(exact_match_rate, 6),
        "relationship_col": target_col,
    }


def is_valid_config(cfg):
    if cfg["distance_mode"] == "gower":
        if cfg["cat_distance_metric"] != "hamming":
            return False
        if cfg["scaler_method"] != "minmax":
            return False
        if cfg["num_distance_metric"] != "euclidean":
            return False
        if float(cfg["num_weight"]) != 1.0 or float(cfg["cat_weight"]) != 1.0:
            return False
    if cfg["num_distance_metric"] != "minkowski" and int(cfg["minkowski_p"]) != 3:
        return False
    return True


def config_to_key(cfg):
    k = int(cfg["k_anonymity"])
    mode = cfg["distance_mode"]
    if mode == "gower":
        return (k, mode)
    return (
        k, cfg["scaler_method"], mode,
        cfg["num_distance_metric"], cfg["cat_distance_metric"], int(cfg["minkowski_p"]),
        float(cfg["num_weight"]), float(cfg["cat_weight"]),
    )


def make_folder_name(idx, cfg):
    cat_tag = f"-cat-{cfg['cat_distance_metric']}" if cfg["distance_mode"] == "weighted_sum" else ""
    return (
        f"{idx:03d}_k{int(cfg['k_anonymity'])}_kn{int(cfg['k_neighbors'])}_cat-{cfg['cat_gen_method']}_"
        f"num-{cfg['num_gen_method']}_scale-{cfg['scaler_method']}_{cfg['distance_mode']}-"
        f"{cfg['num_distance_metric']}{cat_tag}_w-{cfg['distance_profile']}"
    )


def make_display_name(cfg):
    cat_m = cfg["cat_distance_metric"] if cfg["distance_mode"] == "weighted_sum" else "gower"
    return (
        f"k={int(cfg['k_anonymity'])} kn={int(cfg['k_neighbors'])} "
        f"cat={cfg['cat_gen_method']} num={cfg['num_gen_method']} "
        f"scale={cfg['scaler_method']} {cfg['distance_mode']}/{cfg['num_distance_metric']}/{cat_m}/{cfg['distance_profile']}"
    )


def finalize_ranking(rows):
    ranking = pd.DataFrame(rows)
    if ranking.empty:
        ranking["rank"] = []
        return ranking

    relationship_score = 1.0 - (ranking["mean_corr_drift"] / 0.05).clip(lower=0.0, upper=1.0)
    ks_quality = 1.0 - (ranking["mean_ks"] / 0.10).clip(lower=0.0, upper=1.0)
    ranking["relationship_score"] = relationship_score.round(4)
    ranking["ks_quality"] = ks_quality.round(4)
    ranking["quality_score"] = (
        0.33 * ranking["tvd_pass_rate"]
        + 0.33 * ranking["ks_pass_rate"]
        + 0.20 * relationship_score
        + 0.14 * ks_quality
    ).round(4)
    runtime_norm = ranking["runtime_sec"] / max(float(ranking["runtime_sec"].max()), 1e-6)
    ranking["composite_score"] = (ranking["quality_score"] - 0.02 * runtime_norm).round(4)
    ranking = ranking.sort_values(
        by=["overall_pass", "composite_score", "mean_corr_drift", "runtime_sec"],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)
    ranking["rank"] = ranking.index + 1
    return ranking


def row_to_config(row):
    cfg = {
        "k_anonymity": int(row["k_anonymity"]),
        "k_neighbors": int(row["k_neighbors"]),
        "cat_gen_method": str(row["cat_gen_method"]),
        "num_gen_method": str(row["num_gen_method"]),
        "scaler_method": str(row["scaler_method"]),
        "distance_mode": str(row["distance_mode"]),
        "num_distance_metric": str(row["num_distance_metric"]),
        "minkowski_p": int(row["minkowski_p"]),
        "cat_distance_metric": str(row["cat_distance_metric"]),
        "num_weight": float(row["num_weight"]),
        "cat_weight": float(row["cat_weight"]),
        "distance_profile": str(row["distance_profile"]),
        "random_state": RANDOM_STATE,
        "row_synthesis_mode": str(row.get("row_synthesis_mode", ROW_SYNTHESIS_MODE)),
    }
    cfg["target_gen_method"] = str(row.get("target_gen_method", "probability"))
    return cfg


def build_distribution_summary(df_actual, df_out):
    dist_rows = []
    for col in NUMERICAL_COLS:
        actual = pd.to_numeric(df_actual[col], errors="coerce")
        synth = pd.to_numeric(df_out[col], errors="coerce")
        actual_mean = float(actual.mean())
        synth_mean = float(synth.mean())
        dist_rows.append({
            "column": col,
            "type": "numerical",
            "actual_mean": round(actual_mean, 4),
            "synthetic_mean": round(synth_mean, 4),
            "actual_std": round(float(actual.std()), 4),
            "synthetic_std": round(float(synth.std()), 4),
            "mean_diff_pct": round(abs(synth_mean - actual_mean) / max(abs(actual_mean), 1e-6) * 100, 2),
        })
    for col in TVD_CATEGORICAL_COLS:
        if col not in df_actual.columns:
            continue
        actual_counts = df_actual[col].astype(str).value_counts(normalize=True)
        synth_counts = df_out[col].astype(str).value_counts(normalize=True)
        dist_rows.append({
            "column": col,
            "type": "categorical",
            "actual_category_count": int(actual_counts.shape[0]),
            "synthetic_category_count": int(synth_counts.shape[0]),
            "actual_top_category_pct": round(float(actual_counts.iloc[0]) * 100, 2),
            "synthetic_top_category_pct": round(float(synth_counts.iloc[0]) * 100, 2),
            "mean_diff_pct": np.nan,
        })
    return pd.DataFrame(dist_rows)


def _distribution_summary(metrics, df_actual, df_out):
    dist_summary = metrics.get("distribution_summary")
    if isinstance(dist_summary, pd.DataFrame):
        return dist_summary
    return build_distribution_summary(df_actual, df_out)


def display_metrics_report(df_actual, df_out, metrics):
    from IPython.display import display

    karabo_summary = metrics.get("karabo_summary") or {}
    print("=" * 72)
    print("VALIDATION METRICS REPORT")
    print("=" * 72)
    if karabo_summary:
        print(
            f"Real AUC: {karabo_summary.get('real_auc')} | "
            f"Synthetic AUC: {karabo_summary.get('synthetic_auc')} | "
            f"AUC retention: {karabo_summary.get('auc_retention_ratio')} | "
            f"Mean IV retention: {karabo_summary.get('mean_iv_retention')} | "
            f"Target rate drift: {karabo_summary.get('target_rate_drift')}"
        )

    dist_summary = _distribution_summary(metrics, df_actual, df_out)

    sections = [
        ("Pipeline scorecard", metrics.get("scorecard")),
        ("Karabo scorecard", metrics.get("karabo_scorecard")),
        ("Distribution summary", dist_summary),
        ("Categorical distribution (TVD)", metrics.get("category_metrics")),
        ("Numerical distribution (KS / PSI)", metrics.get("numeric_metrics")),
        ("Rare categories", metrics.get("rare_category_metrics")),
        ("Target-feature relationships", metrics.get("relationship_metrics")),
        ("Numerical correlation drift", metrics.get("num_correlation_metrics")),
        ("Categorical pair relationships", metrics.get("cat_pair_relationship_metrics")),
        ("Categorical-numerical relationships", metrics.get("cat_num_relationship_metrics")),
        ("Information Value (IV)", metrics.get("iv_metrics")),
        ("Target rate", metrics.get("target_metrics")),
        ("Model performance (AUC / accuracy / F1)", metrics.get("utility_metrics")),
        ("Privacy", metrics.get("privacy_metrics")),
    ]
    for title, table in sections:
        if isinstance(table, pd.DataFrame) and len(table):
            print(f"\n--- {title} ---")
            display(table)


def plot_validation_charts(df_actual, df_out, metrics):
    karabo_summary = metrics.get("karabo_summary") or {}
    roc_curves = metrics.get("roc_curves") or {}

    if NUMERICAL_COLS:
        n = len(NUMERICAL_COLS)
        ncols = min(4, n)
        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
        axes_flat = np.atleast_1d(axes).ravel()
        for ax, col in zip(axes_flat, NUMERICAL_COLS):
            sns.kdeplot(pd.to_numeric(df_actual[col], errors="coerce"), ax=ax, label="actual", color="steelblue")
            sns.kdeplot(pd.to_numeric(df_out[col], errors="coerce"), ax=ax, label="synthetic", color="darkorange")
            ax.set_title(col)
            ax.legend(fontsize=8)
        for ax in axes_flat[len(NUMERICAL_COLS):]:
            ax.set_visible(False)
        plt.suptitle("Numerical distributions: actual vs synthetic")
        plt.tight_layout()
        plt.show()

    cat_cols = [c for c in TVD_CATEGORICAL_COLS if c in df_actual.columns]
    if cat_cols:
        fig, axes = plt.subplots(1, len(cat_cols), figsize=(5 * len(cat_cols), 4))
        if len(cat_cols) == 1:
            axes = [axes]
        for ax, col in zip(axes, cat_cols):
            actual_pct = df_actual[col].astype(str).value_counts(normalize=True)
            synth_pct = df_out[col].astype(str).value_counts(normalize=True)
            cats = sorted(set(actual_pct.index) | set(synth_pct.index))
            x = np.arange(len(cats))
            width = 0.35
            ax.bar(x - width / 2, actual_pct.reindex(cats, fill_value=0), width, label="actual", color="steelblue")
            ax.bar(x + width / 2, synth_pct.reindex(cats, fill_value=0), width, label="synthetic", color="darkorange")
            ax.set_xticks(x)
            ax.set_xticklabels(cats, rotation=45, ha="right")
            ax.set_title(col)
            ax.legend(fontsize=8)
        plt.suptitle("Categorical distributions: actual vs synthetic")
        plt.tight_layout()
        plt.show()

    iv_df = metrics.get("iv_metrics")
    if isinstance(iv_df, pd.DataFrame) and len(iv_df):
        plot_iv = iv_df.sort_values("original_iv", ascending=True).tail(10)
        fig, ax = plt.subplots(figsize=(8, max(4, len(plot_iv) * 0.35)))
        y = np.arange(len(plot_iv))
        ax.barh(y - 0.15, plot_iv["original_iv"], height=0.3, label="original IV", color="steelblue")
        ax.barh(y + 0.15, plot_iv["synthetic_iv"], height=0.3, label="synthetic IV", color="darkorange")
        ax.set_yticks(y)
        ax.set_yticklabels(plot_iv["feature"])
        ax.set_xlabel("Information Value")
        ax.set_title("IV retention by feature")
        ax.legend()
        plt.tight_layout()
        plt.show()

    if roc_curves:
        fig, ax = plt.subplots(figsize=(6, 5))
        styles = {"real_train": ("steelblue", "Real train"), "synthetic_train": ("darkorange", "Synthetic train")}
        for key, curve in roc_curves.items():
            color, label = styles.get(key, ("gray", key))
            auc_val = karabo_summary.get("real_auc") if key == "real_train" else karabo_summary.get("synthetic_auc")
            ax.plot(curve["fpr"], curve["tpr"], color=color, label=f"{label} (AUC={auc_val})")
        ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title("ROC curves — model utility (held-out test set)")
        ax.legend()
        plt.tight_layout()
        plt.show()
    elif TARGET_COL:
        print("ROC curves not available (binary target required for utility metrics).")


def export_production_outputs(df_out, metrics, cfg, run_name="Production Run"):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(OUTPUT_DIR / "anonymized_dataset.csv", index=False)
    metrics["category_metrics"].to_csv(OUTPUT_DIR / "category_metrics.csv", index=False)
    metrics["numeric_metrics"].to_csv(OUTPUT_DIR / "numeric_metrics.csv", index=False)
    metrics["relationship_metrics"].to_csv(OUTPUT_DIR / "relationship_metrics.csv", index=False)
    metrics["scorecard"].to_csv(OUTPUT_DIR / "scorecard.csv", index=False)
    if metrics.get("karabo_scorecard") is not None:
        metrics["karabo_scorecard"].to_csv(OUTPUT_DIR / "karabo_scorecard.csv", index=False)
    for key, filename in [
        ("rare_category_metrics", "rare_category_metrics.csv"),
        ("num_correlation_metrics", "num_correlation_metrics.csv"),
        ("cat_pair_relationship_metrics", "cat_pair_relationship_metrics.csv"),
        ("cat_num_relationship_metrics", "cat_num_relationship_metrics.csv"),
        ("iv_metrics", "iv_metrics.csv"),
        ("target_metrics", "target_metrics.csv"),
        ("privacy_metrics", "privacy_metrics.csv"),
        ("utility_metrics", "utility_metrics.csv"),
    ]:
        table = metrics.get(key)
        if isinstance(table, pd.DataFrame) and len(table):
            table.to_csv(OUTPUT_DIR / filename, index=False)
    dist_summary = metrics.get("distribution_summary")
    if isinstance(dist_summary, pd.DataFrame) and len(dist_summary):
        dist_summary.to_csv(OUTPUT_DIR / "distribution_summary.csv", index=False)
    roc_curves = metrics.get("roc_curves")
    if roc_curves:
        with open(OUTPUT_DIR / "roc_curves.json", "w") as f:
            json.dump(roc_curves, f, indent=2)
    summary = {
        "runtime_sec": metrics.get("runtime_sec"),
        "peak_memory_mb": metrics.get("peak_memory_mb"),
        "n_suppressed": metrics.get("n_suppressed"),
        "overall_pass": metrics.get("overall_pass"),
        "relationship_col": metrics.get("relationship_col"),
        "target_col": TARGET_COL,
        "karabo_summary": metrics.get("karabo_summary"),
        "inferred_schema": {
            "id_cols": ID_COLS,
            "categorical_cols": CATEGORICAL_COLS,
            "numerical_cols": NUMERICAL_COLS,
            "kanon_qi_cols": KANON_QI_COLS,
            "generalization": GENERALIZATION,
        },
    }
    with open(OUTPUT_DIR / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    user_config = {
        "name": run_name,
        "relationship_col": RELATIONSHIP_COL,
        "target_col": TARGET_COL,
        "row_synthesis_mode": str(cfg.get("row_synthesis_mode", ROW_SYNTHESIS_MODE)),
        "categorical_cols": CATEGORICAL_COLS,
        "target_gen_method": str(cfg.get("target_gen_method", "probability")),
        "numerical_cols": NUMERICAL_COLS,
        "k_anonymity": int(cfg["k_anonymity"]),
        "k_neighbors": int(cfg["k_neighbors"]),
        "num_weight": float(cfg["num_weight"]),
        "cat_weight": float(cfg["cat_weight"]),
        "scaler_method": cfg["scaler_method"],
        "cat_gen_method": cfg["cat_gen_method"],
        "num_gen_method": cfg["num_gen_method"],
        "distance_mode": cfg["distance_mode"],
        "num_distance_metric": cfg["num_distance_metric"],
        "cat_distance_metric": cfg["cat_distance_metric"],
        "minkowski_p": int(cfg["minkowski_p"]),
        "random_state": int(cfg.get("random_state", RANDOM_STATE)),
    }
    with open(OUTPUT_DIR / "config.json", "w") as f:
        json.dump(user_config, f, indent=2)
    with open(OUTPUT_DIR / "user_config.json", "w") as f:
        json.dump(user_config, f, indent=2)
    print(f"Exported production outputs → {OUTPUT_DIR}/")


print("Pipeline functions loaded.")



Pipeline functions loaded.


## 1 — Load data


In [5]:
DATA_PATH = find_data_csv(PIPELINE_ROOT, DATA_FILE_OVERRIDE)
df = pd.read_csv(DATA_PATH)
df = df.replace(r"^\s*$", np.nan, regex=True)
df = df.drop(columns=[c for c in df.columns if df[c].isna().all()]).reset_index(drop=True)

schema = infer_dataset_schema(df, TARGET_COL_OVERRIDE, ID_COLS_OVERRIDE)
apply_schema(schema)

print(f"Loaded {len(df):,} rows from {DATA_PATH.name}")
print(f"ID columns (excluded): {ID_COLS}")
print(f"Categorical QI (features): {CATEGORICAL_COLS}")
print(f"Target column: {TARGET_COL}")
print(f"Numerical QI: {NUMERICAL_COLS}")
print(f"Row synthesis mode: {ROW_SYNTHESIS_MODE}")
print(f"k-anonymity columns: {KANON_QI_COLS}")
print(f"Generalization rules: {GENERALIZATION}")
pd.DataFrame({"role": ["id", "categorical", "numerical", "target", "k-anon"], "columns": [ID_COLS, CATEGORICAL_COLS, NUMERICAL_COLS, [TARGET_COL] if TARGET_COL else [], KANON_QI_COLS]})


Loaded 10,000 rows from Bank Customer Churn Prediction.csv
ID columns (excluded): ['customer_id']
Categorical QI (features): ['country', 'gender']
Target column: churn
Numerical QI: ['credit_score', 'age', 'tenure', 'balance', 'products_number', 'credit_card', 'active_member', 'estimated_salary']
Row synthesis mode: donor
k-anonymity columns: ['country', 'gender', 'age', 'tenure', 'products_number', 'credit_card', 'active_member']
Generalization rules: {'age': {'type': 'quantile_bins', 'q': 20}, 'tenure': {'type': 'bin_floor', 'step': 1}}


,role,columns
0,id,[customer_id]
1,categorical,"[country, gender]"
2,numerical,"[credit_score, age, tenure, balance, products_..."
3,target,[churn]
4,k-anon,"[country, gender, age, tenure, products_number..."


## 2 — Run experiments


In [6]:
suppression_cache = {}
prep_cache = {}
neighbor_cache_store = {}
ranking_rows = []

if EXPERIMENT_RANKING_CSV.exists():
    existing = pd.read_csv(EXPERIMENT_RANKING_CSV)
    done_folders = set(existing["folder"].astype(str))
    ranking_rows = existing.to_dict("records")
    print(f"Resuming: {len(done_folders)} configs already in {EXPERIMENT_RANKING_CSV.name}")
else:
    done_folders = set()

if RUN_MODE == "single":
    combos = pd.DataFrame([{
        "k_anonymity": K_ANONYMITY,
        "k_neighbors": K_NEIGHBORS,
        "cat_gen_method": CAT_GEN_METHOD,
        "num_gen_method": NUM_GEN_METHOD,
        "scaler_method": SCALER_METHOD,
        "distance_mode": DISTANCE_MODE,
        "num_distance_metric": NUM_DISTANCE_METRIC,
        "minkowski_p": MINKOWSKI_P,
        "cat_distance_metric": CAT_DISTANCE_METRIC,
        "num_weight": NUM_WEIGHT,
        "cat_weight": CAT_WEIGHT,
        "distance_profile": "balanced",
        "target_gen_method": TARGET_GEN_METHOD,
    }])
else:
    combos = pd.read_csv(PARAMETER_COMBINATIONS_CSV)
    if len(combos) == 0:
        raise ValueError(f"No parameter rows in {PARAMETER_COMBINATIONS_CSV.name}. Populate parameter_combinations.csv first.")
    end = len(combos) if BATCH_LIMIT is None else min(len(combos), BATCH_START + BATCH_LIMIT)
    combos = combos.iloc[BATCH_START:end].reset_index(drop=True)
    print(f"Batch mode: {len(combos):,} configs from {PARAMETER_COMBINATIONS_CSV.name}")

for i, row in combos.iterrows():
    cfg = row_to_config(row)
    folder = make_folder_name(i + 1 + BATCH_START, cfg)
    if folder in done_folders:
        continue

    print(f"[{i+1}/{len(combos)}] {folder}")
    tracemalloc.start()
    t0 = time.perf_counter()

    k = int(cfg["k_anonymity"])
    if k not in suppression_cache:
        suppression_cache[k] = identify_suppressed(df, k)
    suppressed, pool_idx, synth_idx = suppression_cache[k]

    scaler = cfg["scaler_method"]
    if scaler not in prep_cache:
        prep_cache[scaler] = fit_preprocessors(df, scaler)
    prep = prep_cache[scaler]

    nkey = config_to_key(cfg)
    if nkey not in neighbor_cache_store:
        neighbor_cache_store[nkey] = build_neighbor_cache(cfg, prep, pool_idx, synth_idx)
    neighbor_cache = neighbor_cache_store[nkey]

    df_out = synthesize_dataset(
        df, cfg, prep, suppressed, pool_idx, synth_idx, neighbor_cache,
    )
    metrics = compute_metrics(
        df, df_out, suppressed, RELATIONSHIP_COL,
    )

    runtime_sec = round(time.perf_counter() - t0, 3)
    _, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    ranking_rows.append({
        "folder": folder,
        "name": make_display_name(cfg),
        "relationship_col": RELATIONSHIP_COL,
        "k_anonymity": k,
        "k_neighbors": int(cfg["k_neighbors"]),
        "cat_gen_method": cfg["cat_gen_method"],
        "num_gen_method": cfg["num_gen_method"],
        "target_gen_method": cfg.get("target_gen_method"),
        "scaler_method": scaler,
        "num_weight": float(cfg["num_weight"]),
        "cat_weight": float(cfg["cat_weight"]),
        "distance_profile": cfg["distance_profile"],
        "distance_mode": cfg["distance_mode"],
        "num_distance_metric": cfg["num_distance_metric"],
        "cat_distance_metric": cfg["cat_distance_metric"],
        "minkowski_p": int(cfg["minkowski_p"]),
        "random_state": int(cfg.get("random_state", RANDOM_STATE)),
        "runtime_sec": runtime_sec,
        "peak_memory_mb": round(peak_mem / 1024 / 1024, 2),
        "n_suppressed": int(suppressed.sum()),
        "tvd_pass_rate": metrics["tvd_pass_rate"],
        "ks_pass_rate": metrics["ks_pass_rate"],
        "mean_tvd": metrics["mean_tvd"],
        "mean_ks": metrics["mean_ks"],
        "mean_corr_drift": metrics["mean_corr_drift"],
        "exact_match_rate": metrics["exact_match_rate"],
        "mean_psi": metrics.get("karabo_summary", {}).get("mean_psi"),
        "target_rate_drift": metrics.get("karabo_summary", {}).get("target_rate_drift"),
        "auc_retention_ratio": metrics.get("karabo_summary", {}).get("auc_retention_ratio"),
        "real_auc": metrics.get("karabo_summary", {}).get("real_auc"),
        "synthetic_auc": metrics.get("karabo_summary", {}).get("synthetic_auc"),
        "mean_iv_retention": metrics.get("karabo_summary", {}).get("mean_iv_retention"),
        "overall_pass": metrics["overall_pass"],
    })
    done_folders.add(folder)

    if len(ranking_rows) % CHECKPOINT_EVERY == 0:
        finalize_ranking(ranking_rows).to_csv(EXPERIMENT_RANKING_CSV, index=False)
        print(f"  checkpoint → {EXPERIMENT_RANKING_CSV}")

    if RUN_MODE == "single":
        last_df_out = df_out
        last_metrics = metrics
        last_metrics["runtime_sec"] = runtime_sec
        last_metrics["peak_memory_mb"] = round(peak_mem / 1024 / 1024, 2)
        last_metrics["n_suppressed"] = int(suppressed.sum())
        last_cfg = cfg

ranking = finalize_ranking(ranking_rows)
ranking.to_csv(EXPERIMENT_RANKING_CSV, index=False)
print(f"\nSaved {len(ranking):,} rows → {EXPERIMENT_RANKING_CSV}")
if len(ranking):
    print(f"Top: {ranking.iloc[0]['folder']} | overall_pass={ranking.iloc[0]['overall_pass']}")
    print(f"Passing all checks: {int(ranking['overall_pass'].sum())}/{len(ranking)}")
ranking.head(10)


Resuming: 11 configs already in experiment_ranking.csv
Batch mode: 80 configs from parameter_combinations.csv
[11/80] 011_k3_kn15_cat-probability_num-interpolation_scale-standard_weighted_sum-euclidean-cat-hamming_w-balanced


c:\Users\admin\OneDrive\Documents\GitHub\KNN_demo\.venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
c:\Users\admin\OneDrive\Documents\GitHub\KNN_demo\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[12/80] 012_k3_kn15_cat-probability_num-weighted_mean_scale-standard_weighted_sum-euclidean-cat-hamming_w-balanced


c:\Users\admin\OneDrive\Documents\GitHub\KNN_demo\.venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
c:\Users\admin\OneDrive\Documents\GitHub\KNN_demo\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[13/80] 013_k5_kn3_cat-mode_num-interpolation_scale-standard_weighted_sum-euclidean-cat-hamming_w-balanced


...

c:\Users\admin\OneDrive\Documents\GitHub\KNN_demo\.venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
c:\Users\admin\OneDrive\Documents\GitHub\KNN_demo\.venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
c:\Users\admin\OneDrive\Documents\GitHub\KNN_demo\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-l

[67/80] 067_k3_kn15_cat-probability_num-interpolation_scale-minmax_gower-euclidean_w-balanced


c:\Users\admin\OneDrive\Documents\GitHub\KNN_demo\.venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
c:\Users\admin\OneDrive\Documents\GitHub\KNN_demo\.venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
c:\Users\admin\OneDrive\Documents\GitHub\KNN_demo\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-l

[68/80] 068_k3_kn15_cat-probability_num-weighted_mean_scale-minmax_gower-euclidean_w-balanced


KeyboardInterrupt: 

## 3 — Export best config to `output/` (optional)


In [ ]:
report_df_actual = df
report_df_out = None
report_metrics = None

if RUN_MODE == "single":
    export_production_outputs(last_df_out, last_metrics, last_cfg, RUN_NAME)
    report_df_out = last_df_out
    report_metrics = last_metrics
elif EXPORT_TOP_TO_OUTPUT and len(ranking):
    passing = ranking[ranking["overall_pass"] == True]
    top = passing.iloc[0] if len(passing) else ranking.iloc[0]
    if len(passing) == 0:
        print("Warning: no passing configs — exporting rank 1 anyway")
    top_cfg = {
        "k_anonymity": int(top["k_anonymity"]),
        "k_neighbors": int(top["k_neighbors"]),
        "cat_gen_method": top["cat_gen_method"],
        "num_gen_method": top["num_gen_method"],
        "target_gen_method": top.get("target_gen_method"),
        "scaler_method": top["scaler_method"],
        "distance_mode": top["distance_mode"],
        "num_distance_metric": top["num_distance_metric"],
        "minkowski_p": int(top["minkowski_p"]),
        "cat_distance_metric": top["cat_distance_metric"],
        "num_weight": float(top["num_weight"]),
        "cat_weight": float(top["cat_weight"]),
        "distance_profile": top["distance_profile"],
        "random_state": int(top["random_state"]),
        "row_synthesis_mode": str(top.get("row_synthesis_mode", ROW_SYNTHESIS_MODE)),
    }
    k = int(top_cfg["k_anonymity"])
    suppressed, pool_idx, synth_idx = identify_suppressed(df, k)
    prep = fit_preprocessors(df, top_cfg["scaler_method"])
    ncache = build_neighbor_cache(top_cfg, prep, pool_idx, synth_idx)
    df_top = synthesize_dataset(
        df, top_cfg, prep, suppressed, pool_idx, synth_idx, ncache,
    )
    top_metrics = compute_metrics(
        df, df_top, suppressed, RELATIONSHIP_COL,
    )
    top_metrics["runtime_sec"] = float(top["runtime_sec"])
    top_metrics["peak_memory_mb"] = float(top["peak_memory_mb"])
    top_metrics["n_suppressed"] = int(top["n_suppressed"])
    export_production_outputs(df_top, top_metrics, top_cfg, run_name=str(top["name"]))
    print(f"Exported config (rank {int(top['rank'])}): {top['folder']} | overall_pass={top['overall_pass']}")
    report_df_out = df_top
    report_metrics = top_metrics


## 4 — Validation metrics report

Displays distribution, rare categories, relationships, IV, model performance (AUC-ROC), and scorecards for the exported run.

In [ ]:
if report_metrics is None and (OUTPUT_DIR / "anonymized_dataset.csv").exists():
    report_df_out = pd.read_csv(OUTPUT_DIR / "anonymized_dataset.csv")
    k = int(json.loads((OUTPUT_DIR / "config.json").read_text()).get("k_anonymity", 5))
    suppressed, _, _ = identify_suppressed(report_df_actual, k)
    report_metrics = compute_metrics(report_df_actual, report_df_out, suppressed, RELATIONSHIP_COL)
    print(f"Loaded exported outputs from {OUTPUT_DIR}/")

if report_metrics is not None and report_df_out is not None:
    display_metrics_report(report_df_actual, report_df_out, report_metrics)
    plot_validation_charts(report_df_actual, report_df_out, report_metrics)
else:
    print("No exported run available — run section 3 first (single mode or EXPORT_TOP_TO_OUTPUT).")